# Wave Equation with NumPy

Evolve the one-dimensional wave equation directly in NumPy before asking NRPy to generate C.

Navigation: [Index](../index.ipynb) | Previous: [Reference Metrics](../1-intro/reference_metric.ipynb) | Next: [Method of Lines and Runge-Kutta](method_of_lines_and_rk.ipynb)

## Why This Matters
Before using generated C, it helps to see a full evolution in ordinary arrays: ghost zones provide neighboring values, and RK stages advance time.

## What You Will See
You will inspect printed expressions, generated artifacts, or diagnostic tables and then connect them to the next notebook.

Prerequisite bridge: this notebook follows [Reference Metrics](../1-intro/reference_metric.ipynb). Terms are defined here before they are used.

## Evolve a Periodic Wave
Ghost zones are extra cells around the domain that make stencil reads near boundaries use the same formula as interior reads. A Courant factor sets the time step as a fraction of the grid spacing.

In [1]:
import numpy as np

final_time = 0.2
courant_factor = 0.25
rows = []

for npoints in [64, 128, 256]:
    length = 2.0 * np.pi
    dx = length / npoints
    dt = courant_factor * dx
    steps = int(np.ceil(final_time / dt))
    dt = final_time / steps
    x = dx * np.arange(npoints)
    u = np.sin(x)
    v = np.zeros_like(u)

    for step in range(steps):
        padded = np.pad(u, 2, mode="wrap")
        lap = (-padded[0:-4] + 16.0 * padded[1:-3] - 30.0 * padded[2:-2] + 16.0 * padded[3:-1] - padded[4:]) / (12.0 * dx * dx)
        k1u = v
        k1v = lap

        u2 = u + 0.5 * dt * k1u
        v2 = v + 0.5 * dt * k1v
        padded = np.pad(u2, 2, mode="wrap")
        lap = (-padded[0:-4] + 16.0 * padded[1:-3] - 30.0 * padded[2:-2] + 16.0 * padded[3:-1] - padded[4:]) / (12.0 * dx * dx)
        k2u = v2
        k2v = lap

        u3 = u + 0.5 * dt * k2u
        v3 = v + 0.5 * dt * k2v
        padded = np.pad(u3, 2, mode="wrap")
        lap = (-padded[0:-4] + 16.0 * padded[1:-3] - 30.0 * padded[2:-2] + 16.0 * padded[3:-1] - padded[4:]) / (12.0 * dx * dx)
        k3u = v3
        k3v = lap

        u4 = u + dt * k3u
        v4 = v + dt * k3v
        padded = np.pad(u4, 2, mode="wrap")
        lap = (-padded[0:-4] + 16.0 * padded[1:-3] - 30.0 * padded[2:-2] + 16.0 * padded[3:-1] - padded[4:]) / (12.0 * dx * dx)
        k4u = v4
        k4v = lap

        u = u + dt * (k1u + 2.0 * k2u + 2.0 * k3u + k4u) / 6.0
        v = v + dt * (k1v + 2.0 * k2v + 2.0 * k3v + k4v) / 6.0

    exact_u = np.sin(x) * np.cos(final_time)
    error = float(np.max(np.abs(u - exact_u)))
    rows.append((npoints, dx, steps, error))

print("npoints dx steps max_error convergence_factor")
previous_error = None
for npoints, dx, steps, error in rows:
    factor = "" if previous_error is None else f"{previous_error / error:.3f}"
    print(npoints, f"{dx:.6e}", steps, f"{error:.6e}", factor)
    previous_error = error

assert rows[-1][3] < rows[0][3]

npoints dx steps max_error convergence_factor
64 9.817477e-02 9 2.056205e-08 
128 4.908739e-02 17 1.287406e-09 15.972
256 2.454369e-02 33 8.053413e-11 15.986


## Interpreting the Output
The diagnostic table shows a stable finite-difference wave update in plain NumPy. NRPy automates the same mathematical pieces when later notebooks replace hand-written loops with generated kernels.

## Where This Leads
- [Method of Lines and Runge-Kutta](method_of_lines_and_rk.ipynb)
- [Boundary Conditions and Convergence](boundary_conditions_and_convergence.ipynb)
- [Wave Equation and C Code Generation](../3-wave_equation/wave_equation_and_c_codegen.ipynb)